In [1]:
import warnings
warnings.filterwarnings("ignore")


from pathlib import Path

import numpy as np
import pandas as pd

import scipy.sparse

import optuna
import joblib


from catboost import CatBoostClassifier


from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score
)

In [2]:
DATA_PATH = Path("../data/processed")
MODEL_PATH = Path("../models")

In [3]:
X_train = scipy.sparse.load_npz(
    DATA_PATH / "X_train.npz"
)

X_test = scipy.sparse.load_npz(
    DATA_PATH / "X_test.npz"
)


y_train = np.load(
    DATA_PATH / "y_train.npy"
)


y_test = np.load(
    DATA_PATH / "y_test.npy"
)

In [4]:
def calculate_metrics(
    y_true,
    probability,
    threshold=0.5
):

    prediction = (
        probability >= threshold
    ).astype(int)


    auc = roc_auc_score(
        y_true,
        probability
    )


    gini = (
        2 * auc
        - 1
    )


    return {

        "ROC_AUC": auc,

        "GINI": gini,

        "Recall":
            recall_score(
                y_true,
                prediction
            ),

        "Precision":
            precision_score(
                y_true,
                prediction
            ),

        "F1":
            f1_score(
                y_true,
                prediction
            )
    }

In [5]:
def objective(trial):

    params = {

        "iterations":
            trial.suggest_int(
                "iterations",
                300,
                1000
            ),


        "depth":
            trial.suggest_int(
                "depth",
                4,
                10
            ),


        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.2,
                log=True
            ),


        "l2_leaf_reg":
            trial.suggest_float(
                "l2_leaf_reg",
                1,
                10
            ),


        "random_strength":
            trial.suggest_float(
                "random_strength",
                0.1,
                5
            ),


        "border_count":
            trial.suggest_int(
                "border_count",
                32,
                255
            ),


        "loss_function":
            "Logloss",


        "eval_metric":
            "AUC",


        "auto_class_weights":
            "Balanced",


        "verbose":
            False,


        "random_state":
            42
    }


    model = CatBoostClassifier(
        **params
    )


    model.fit(
        X_train,
        y_train
    )


    probability = model.predict_proba(
        X_test
    )[:,1]


    auc = roc_auc_score(
        y_test,
        probability
    )


    return auc

In [6]:
study = optuna.create_study(
    direction="maximize"
)


study.optimize(
    objective,
    n_trials=50
)

[I 2026-07-26 16:00:01,034] A new study created in memory with name: no-name-6f0a4fc1-ae60-493d-bc97-dc9e0779ebc6
[I 2026-07-26 16:00:02,271] Trial 0 finished with value: 0.8336176237634013 and parameters: {'iterations': 560, 'depth': 4, 'learning_rate': 0.03327565472555654, 'l2_leaf_reg': 6.733408289582128, 'random_strength': 2.761023572270387, 'border_count': 238}. Best is trial 0 with value: 0.8336176237634013.
[I 2026-07-26 16:00:04,258] Trial 1 finished with value: 0.8002637559468035 and parameters: {'iterations': 726, 'depth': 6, 'learning_rate': 0.11176724299788907, 'l2_leaf_reg': 1.332156989140397, 'random_strength': 3.5746630994873327, 'border_count': 213}. Best is trial 0 with value: 0.8336176237634013.
[I 2026-07-26 16:00:07,590] Trial 2 finished with value: 0.8216178929549467 and parameters: {'iterations': 648, 'depth': 8, 'learning_rate': 0.023920406090215663, 'l2_leaf_reg': 3.043520828849879, 'random_strength': 0.2024907939961757, 'border_count': 180}. Best is trial 0 wit

In [7]:
study.best_params

{'iterations': 793,
 'depth': 4,
 'learning_rate': 0.012266970473659121,
 'l2_leaf_reg': 7.243065440605548,
 'random_strength': 2.056941604305263,
 'border_count': 236}

In [8]:
best_params = study.best_params


final_model = CatBoostClassifier(

    **best_params,

    loss_function="Logloss",

    eval_metric="AUC",

    auto_class_weights="Balanced",

    random_state=42,

    verbose=False
)


final_model.fit(
    X_train,
    y_train
)

CatBoostClassifier(auto_class_weights='Balanced', border_count=236, depth=4, eval_metric='AUC', iterations=793, l2_leaf_reg=7.243065440605548, learning_rate=0.012266970473659121, loss_function='Logloss', random_state=42, random_strength=2.056941604305263, verbose=False)

In [9]:
probability = final_model.predict_proba(
    X_test
)[:,1]


metrics = calculate_metrics(
    y_test,
    probability
)


metrics

{'ROC_AUC': 0.8396187833577504,
 'GINI': 0.6792375667155008,
 'Recall': 0.8048128342245989,
 'Precision': 0.4942528735632184,
 'F1': 0.612410986775178}

In [10]:
joblib.dump(
    final_model,
    MODEL_PATH / "churn_catboost_tuned.pkl"
)

['..\\models\\churn_catboost_tuned.pkl']